In [ ]:
"""
=============================================================
  ♔ CHESS BATTLE — PARAMETER OPTIMIZATION BOT
  ─────────────────────────────────────────────────────────
  Parameters tuned:
    ♔ KING   : king_cash_guard_pct, king_total_stop_pct
    ♝ BISHOP : stop_loss_pct, atr_stop_mult, max_drawdown_pct
    ♜ ROOK   : max_positions, top_positions, top_allocation_pct
    ♟ PAWN   : promotion_threshold
    ♞ KNIGHT : nifty_drop_trigger, hedge_allocation_pct
    ♛ QUEEN  : atr_period

  Output: chess_battle_optimization.xlsx
    Sheet 1 — All Results
    Sheet 2 — Top 20 Win Rate
    Sheet 3 — Top 20 Sharpe
    Sheet 4 — Top 20 Final Capital
    Sheet 5 — Top 20 CAGR
    Sheet 6 — Parameter Impact
    Sheet 7 — Best Combo Summary
=============================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import product

# ═══════════════════════════════════════════════════════════
#  FIXED SETTINGS (not tuned)
# ═══════════════════════════════════════════════════════════
INITIAL_CAPITAL      = 100_000
LOOKBACK_PERIOD      = "10y"
NIFTY_TICKER         = "^NSEI"
PROMOTION_SELL_PCT   = 0.50       # always sell 50% on promotion
HEDGE_RECOVERY_DAYS  = 14
MAX_WORKERS          = 20
CACHE_FILE           = "price_cache.pkl"
USE_CACHE            = True

# ═══════════════════════════════════════════════════════════
#  PARAMETER GRID  ← edit values here
#  Total combos = product of all list lengths
#  Keep total under 500 for reasonable runtime
# ═══════════════════════════════════════════════════════════

# ♔ KING
KING_CASH_GUARD_LIST   = [0.10, 0.20, 0.30]          # 3
KING_TOTAL_STOP_LIST   = [0.08, 0.12, 0.18]          # 3

# ♝ BISHOP
STOP_LOSS_LIST         = [0.05, 0.08, 0.12]          # 3
ATR_MULT_LIST          = [2.0,  3.0,  4.0]           # 3
MAX_DRAWDOWN_LIST      = [0.06, 0.09, 0.12]          # 3

# ♜ ROOK
MAX_POSITIONS_LIST     = [5,    8,    12]             # 3
TOP_ALLOC_LIST         = [0.50, 0.65, 0.80]          # 3

# ♟ PAWN
PROMOTION_LIST         = [0.30, 0.50, 0.75]          # 3

# ♞ KNIGHT
NIFTY_DROP_LIST        = [-0.03, -0.05, -0.08]       # 3
HEDGE_ALLOC_LIST       = [0.10,  0.20,  0.30]        # 3

# ♛ QUEEN
ATR_PERIOD_LIST        = [14, 20, 28]                # 3

total_combos = (
    len(KING_CASH_GUARD_LIST)
    * len(KING_TOTAL_STOP_LIST)
    * len(STOP_LOSS_LIST)
    * len(ATR_MULT_LIST)
    * len(MAX_DRAWDOWN_LIST)
    * len(MAX_POSITIONS_LIST)
    * len(TOP_ALLOC_LIST)
    * len(PROMOTION_LIST)
    * len(NIFTY_DROP_LIST)
    * len(HEDGE_ALLOC_LIST)
    * len(ATR_PERIOD_LIST)
)

print("=" * 60)
print("  ♔ CHESS BATTLE OPTIMIZER")
print(f"  Total combinations : {total_combos}")
print(f"  Estimated time     : ~{total_combos * 8 / 3600:.1f} hrs")
print("=" * 60 + "\n")

# ═══════════════════════════════════════════════════════════
#  HELPERS
# ═══════════════════════════════════════════════════════════
def normalize_index(df):
    idx = df.index
    if hasattr(idx, "tz") and idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    df.index = idx.normalize()
    return df

def load_nse_tickers():
    df      = pd.read_csv("data/EQUITY_L.csv")
    symbols = df["SYMBOL"].dropna().astype(str).str.strip().unique().tolist()
    return [s + ".NS" for s in symbols if "&" not in s]

# ═══════════════════════════════════════════════════════════
#  ♛ QUEEN — UTBot (parameterised by atr_period)
# ═══════════════════════════════════════════════════════════
def compute_utbot(df, atr_period):
    df    = df.copy()
    close = df["Close"]
    high  = df["High"]
    low   = df["Low"]
    tr    = np.maximum(
        high - low,
        np.maximum(
            (high - close.shift()).abs(),
            (low  - close.shift()).abs()
        )
    )
    df["atr"]            = tr.rolling(atr_period).mean()
    df["upper"]          = close - df["atr"]
    df["lower"]          = close + df["atr"]
    df["volatility"]     = close.pct_change(fill_method=None).rolling(20).std()
    df["momentum"]       = close.pct_change(fill_method=None).rolling(10).mean()
    df["trend_strength"] = (
        (close - close.rolling(20).mean())
        / close.rolling(20).std()
    )

    trend = [1]
    for i in range(1, len(df)):
        if   close.iloc[i] > df["lower"].iloc[i-1]: trend.append(1)
        elif close.iloc[i] < df["upper"].iloc[i-1]: trend.append(-1)
        else:                                        trend.append(trend[-1])

    df["trend"] = trend
    df["buy"]   = (df["trend"] == 1)  & (df["trend"].shift() == -1)
    df["sell"]  = (df["trend"] == -1) & (df["trend"].shift() == 1)
    return df

# ═══════════════════════════════════════════════════════════
#  LOAD RAW DATA ONCE  (OHLCV only, no signals yet)
#  Signals recomputed per atr_period during optimization
# ═══════════════════════════════════════════════════════════
def load_raw_data(tickers):
    raw_cache = "raw_chess_opt.pkl"

    if USE_CACHE and os.path.exists(raw_cache):
        print("⚡ Loading raw data cache...")
        with open(raw_cache, "rb") as f:
            data = pickle.load(f)
        print(f"✅ {len(data)} stocks loaded\n")
        return data

    if USE_CACHE and os.path.exists(CACHE_FILE):
        print("📦 Loading yfinance cache...")
        price_data = pd.read_pickle(CACHE_FILE)
    else:
        print(f"⬇️  Downloading {len(tickers)} stocks...")
        t0 = time.time()
        price_data = yf.download(
            tickers, period=LOOKBACK_PERIOD,
            group_by="ticker", auto_adjust=True,
            threads=True, progress=True
        )
        price_data.to_pickle(CACHE_FILE)
        print(f"✅ Downloaded in {(time.time()-t0)/60:.1f} min\n")

    print("⚙️  Extracting OHLCV per stock...")
    raw_data = {}

    def extract(ticker):
        try:
            df = price_data[ticker].dropna()
            if df.empty or len(df) < 60: return None
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = normalize_index(df)
            return ticker, df[["Open","High","Low","Close","Volume"]]
        except: return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for r in ex.map(extract, tickers):
            if r: raw_data[r[0]] = r[1]

    print(f"✅ {len(raw_data)} stocks extracted\n")
    with open(raw_cache, "wb") as f:
        pickle.dump(raw_data, f)
    return raw_data

def load_nifty():
    try:
        nifty = yf.download(NIFTY_TICKER, period=LOOKBACK_PERIOD,
                            auto_adjust=True, progress=False)
        if isinstance(nifty.columns, pd.MultiIndex):
            nifty.columns = nifty.columns.get_level_values(0)
        nifty = normalize_index(nifty)
        nifty["daily_return"] = nifty["Close"].pct_change(fill_method=None)
        nifty["ma5"]          = nifty["Close"].rolling(5).mean()
        return nifty
    except: return None

# ═══════════════════════════════════════════════════════════
#  ♜ ROOK allocator
# ═══════════════════════════════════════════════════════════
def rook_allocate(candidates, deployable_cash, open_count,
                  max_positions, top_positions, top_alloc_pct):
    allocations  = []
    rest_alloc   = 1.0 - top_alloc_pct
    if not candidates or deployable_cash <= 0:
        return allocations

    top  = candidates[:top_positions]
    rest = candidates[top_positions:max_positions - open_count]

    if top:
        per = (deployable_cash * top_alloc_pct) / len(top)
        for ticker, score, row in top:
            allocations.append((ticker, row, per))

    if rest:
        per = (deployable_cash * rest_alloc) / len(rest)
        for ticker, score, row in rest:
            allocations.append((ticker, row, per))

    return allocations

# ═══════════════════════════════════════════════════════════
#  SINGLE BACKTEST
# ═══════════════════════════════════════════════════════════
def run_single(
    all_data, all_dates, nifty,
    king_cash_guard, king_total_stop,
    stop_loss, atr_mult, max_drawdown,
    max_positions, top_positions, top_alloc,
    promotion_threshold,
    nifty_drop_trigger, hedge_alloc,
):
    cash           = float(INITIAL_CAPITAL)
    open_positions = {}
    promoted_pawns = set()
    hedge_active   = False
    hedge_entry    = None
    hedge_capital  = 0.0
    trade_log      = []
    promotion_log  = []
    hedge_pnl      = []
    equity_curve   = []
    king_retreated = False

    for current_date in all_dates:

        # ♔ King total stop
        total_pv = cash
        for t, pos in open_positions.items():
            df = all_data.get(t)
            if df is not None and current_date in df.index:
                total_pv += pos["shares"] * float(df.loc[current_date]["Close"])

        if not king_retreated:
            if (INITIAL_CAPITAL - total_pv) / INITIAL_CAPITAL >= king_total_stop:
                king_retreated = True

        if king_retreated:
            for ticker in list(open_positions.keys()):
                df = all_data.get(ticker)
                if df is not None and current_date in df.index:
                    price  = float(df.loc[current_date]["Close"])
                    pos    = open_positions[ticker]
                    profit = pos["shares"] * price - pos["invested"]
                    cash  += pos["shares"] * price
                    trade_log.append({
                        "profit":     profit,
                        "return_pct": (price/pos["entry_price"]-1)*100,
                        "reason":     "King Retreat",
                        "promoted":   ticker in promoted_pawns,
                        "hold_days":  (current_date - pos["entry_date"]).days,
                    })
            open_positions.clear()
            equity_curve.append(cash)
            continue

        # ♞ Knight hedge
        nifty_ret   = None
        nifty_close = None
        nifty_ma5   = None

        if nifty is not None and current_date in nifty.index:
            nr          = nifty.loc[current_date]
            nifty_ret   = float(nr["daily_return"]) if not pd.isna(nr["daily_return"]) else 0
            nifty_close = float(nr["Close"])
            nifty_ma5   = float(nr["ma5"]) if not pd.isna(nr["ma5"]) else nifty_close

        if (not hedge_active and nifty_ret is not None
                and nifty_ret <= nifty_drop_trigger
                and cash > INITIAL_CAPITAL * hedge_alloc):
            hedge_capital = min(total_pv * hedge_alloc, cash * 0.5)
            hedge_entry   = nifty_close
            hedge_active  = True
            cash         -= hedge_capital

        if (hedge_active and nifty_close is not None
                and nifty_ma5 is not None
                and nifty_close > nifty_ma5
                and hedge_entry is not None):
            h_ret    = (nifty_close - hedge_entry) / hedge_entry
            h_profit = hedge_capital * (-h_ret)
            cash    += hedge_capital + h_profit
            hedge_pnl.append(h_profit)
            hedge_active = False

        # ♝ Bishops — exit check
        for ticker in list(open_positions.keys()):
            df = all_data.get(ticker)
            if df is None or current_date not in df.index: continue
            row = df.loc[current_date]
            pos = open_positions[ticker]

            pos["highest_price"] = max(
                pos["highest_price"], float(row["Close"])
            )

            # ♟ Pawn promotion
            curr_ret = float(row["Close"]) / pos["entry_price"] - 1
            if curr_ret >= promotion_threshold and ticker not in promoted_pawns:
                sell_sh   = pos["shares"] * PROMOTION_SELL_PCT
                sell_val  = sell_sh * float(row["Close"])
                cash     += sell_val
                pos["shares"]   -= sell_sh
                pos["invested"] *= (1 - PROMOTION_SELL_PCT)
                promoted_pawns.add(ticker)
                promotion_log.append(curr_ret)

            stop_p = pos["entry_price"] * (1 - stop_loss)
            atr_stp= pos["highest_price"] - float(row["atr"]) * atr_mult
            dd     = (pos["highest_price"] - float(row["Close"])) / pos["highest_price"]

            if   float(row["Close"]) <= stop_p:  reason = "Stop Loss"
            elif float(row["Close"]) <= atr_stp: reason = "ATR Stop"
            elif dd >= max_drawdown:              reason = "Max DD"
            elif bool(row["sell"]):              reason = "Queen Sell"
            else: continue

            exit_p = float(row["Close"])
            profit = pos["shares"] * exit_p - pos["invested"]
            cash  += pos["shares"] * exit_p

            trade_log.append({
                "profit":     profit,
                "return_pct": (exit_p/pos["entry_price"]-1)*100,
                "reason":     reason,
                "promoted":   ticker in promoted_pawns,
                "hold_days":  (current_date - pos["entry_date"]).days,
            })
            promoted_pawns.discard(ticker)
            del open_positions[ticker]

        # ♛ Queen + ♜ Rook — entry
        slots           = max_positions - len(open_positions)
        king_reserve    = total_pv * king_cash_guard
        deployable_cash = max(cash - king_reserve, 0)

        if slots > 0 and deployable_cash > 0:
            candidates = []
            for ticker, df in all_data.items():
                if ticker in open_positions: continue
                if current_date not in df.index: continue
                row = df.loc[current_date]

                if pd.isna(row["volatility"]) or row["volatility"] == 0: continue
                if not bool(row["buy"]): continue
                if pd.isna(row["momentum"]) or float(row["momentum"]) <= 0: continue
                if pd.isna(row["trend_strength"]) or float(row["trend_strength"]) <= 0: continue

                score = (
                    (1.0 / float(row["volatility"]))
                    * (1 + float(row["momentum"]))
                    * max(float(row["trend_strength"]), 0.1)
                )
                candidates.append((ticker, score, row))

            candidates.sort(key=lambda x: x[1], reverse=True)

            top_pos  = min(top_positions, max_positions)
            alloc_plan = rook_allocate(
                candidates, deployable_cash, len(open_positions),
                max_positions, top_pos, top_alloc
            )

            for ticker, row, allocation in alloc_plan:
                if allocation <= 0 or cash <= 0: break
                price  = float(row["Close"])
                shares = allocation / price
                cash  -= allocation
                open_positions[ticker] = {
                    "entry_date":    current_date,
                    "entry_price":   price,
                    "highest_price": price,
                    "shares":        shares,
                    "invested":      allocation,
                }

        # ♔ King counts wealth
        pv = cash
        for ticker, pos in open_positions.items():
            df = all_data.get(ticker)
            if df is not None and current_date in df.index:
                pv += pos["shares"] * float(df.loc[current_date]["Close"])
        equity_curve.append(pv)

    # ── METRICS ──────────────────────────────────────────
    if not equity_curve or not trade_log:
        return None

    trades  = pd.DataFrame(trade_log)
    eq      = np.array(equity_curve)
    ret_s   = pd.Series(eq).pct_change().dropna()
    years   = len(all_dates) / 252

    sharpe  = (ret_s.mean()/ret_s.std()*np.sqrt(252)) if ret_s.std() > 0 else 0
    peak    = np.maximum.accumulate(eq)
    mdd_r   = ((eq - peak)/peak).min() * 100
    cagr    = ((eq[-1]/INITIAL_CAPITAL)**(1/years)-1)*100 if years > 0 else 0

    wins    = trades[trades["profit"] > 0]
    losses  = trades[trades["profit"] < 0]
    wr      = len(wins)/len(trades)*100
    avg_win = wins["profit"].mean()   if len(wins)   > 0 else 0
    avg_los = losses["profit"].mean() if len(losses) > 0 else 0
    rr      = abs(avg_win/avg_los)    if avg_los != 0   else 0
    reasons = trades["reason"].value_counts().to_dict()
    prom_wr = (
        trades[trades["promoted"]==True]["profit"].gt(0).mean()*100
        if trades["promoted"].any() else 0
    )

    return {
        # ── Parameters ──────────────────────────────
        "♔ king_cash_guard":    king_cash_guard,
        "♔ king_total_stop":    king_total_stop,
        "♝ stop_loss":          stop_loss,
        "♝ atr_multiplier":     atr_mult,
        "♝ max_drawdown":       max_drawdown,
        "♜ max_positions":      max_positions,
        "♜ top_alloc_%":        round(top_alloc*100, 0),
        "♟ promotion_at_%":     round(promotion_threshold*100, 0),
        "♞ nifty_drop_trigger": nifty_drop_trigger,
        "♞ hedge_alloc_%":      round(hedge_alloc*100, 0),
        # ── Results ─────────────────────────────────
        "final_capital_₹":      round(eq[-1], 2),
        "total_return_%":       round((eq[-1]/INITIAL_CAPITAL-1)*100, 2),
        "cagr_%":               round(cagr, 2),
        "sharpe_ratio":         round(sharpe, 3),
        "max_drawdown_%":       round(mdd_r, 2),
        "win_rate_%":           round(wr, 2),
        "avg_win_₹":            round(avg_win, 2),
        "avg_loss_₹":           round(avg_los, 2),
        "risk_reward":          round(rr, 3),
        "total_trades":         len(trades),
        "avg_hold_days":        round(trades["hold_days"].mean(), 1),
        "promoted_win_rate_%":  round(prom_wr, 2),
        "total_promotions":     len(promotion_log),
        "total_hedge_pnl_₹":   round(sum(hedge_pnl), 2),
        "hedge_deployments":    len(hedge_pnl),
        # ── Exit breakdown ──────────────────────────
        "exits_stop_loss":      reasons.get("Stop Loss", 0),
        "exits_atr":            reasons.get("ATR Stop", 0),
        "exits_max_dd":         reasons.get("Max DD", 0),
        "exits_queen_sell":     reasons.get("Queen Sell", 0),
        "exits_king_retreat":   reasons.get("King Retreat", 0),
    }

# ═══════════════════════════════════════════════════════════
#  PARAMETER IMPACT TABLE
# ═══════════════════════════════════════════════════════════
def build_impact(df):
    param_map = {
        "♔ king_cash_guard":    KING_CASH_GUARD_LIST,
        "♔ king_total_stop":    KING_TOTAL_STOP_LIST,
        "♝ stop_loss":          STOP_LOSS_LIST,
        "♝ atr_multiplier":     ATR_MULT_LIST,
        "♝ max_drawdown":       MAX_DRAWDOWN_LIST,
        "♜ max_positions":      MAX_POSITIONS_LIST,
        "♜ top_alloc_%":        [round(v*100,0) for v in TOP_ALLOC_LIST],
        "♟ promotion_at_%":     [round(v*100,0) for v in PROMOTION_LIST],
        "♞ nifty_drop_trigger": NIFTY_DROP_LIST,
        "♞ hedge_alloc_%":      [round(v*100,0) for v in HEDGE_ALLOC_LIST],
    }
    rows = []
    for param, values in param_map.items():
        for val in values:
            sub = df[df[param] == val]
            if sub.empty: continue
            rows.append({
                "Piece":                param.split()[0],
                "Parameter":            param,
                "Value":                val,
                "Avg Win Rate %":       round(sub["win_rate_%"].mean(), 2),
                "Avg Sharpe":           round(sub["sharpe_ratio"].mean(), 3),
                "Avg CAGR %":           round(sub["cagr_%"].mean(), 2),
                "Avg Final Capital ₹":  round(sub["final_capital_₹"].mean(), 0),
                "Avg Max DD %":         round(sub["max_drawdown_%"].mean(), 2),
                "Avg Risk/Reward":      round(sub["risk_reward"].mean(), 3),
                "Avg Trades":           round(sub["total_trades"].mean(), 0),
                "# Combos":             len(sub),
            })
    return pd.DataFrame(rows).sort_values(
        ["Piece","Parameter","Avg Win Rate %"], ascending=[True,True,False]
    )

# ═══════════════════════════════════════════════════════════
#  SAVE EXCEL
# ═══════════════════════════════════════════════════════════
def save_excel(df_results, impact_df, output="chess_battle_optimization.xlsx"):
    print(f"\n💾 Saving {output}...")

    # Strip timezone from any datetime columns
    for col in df_results.select_dtypes(include=["datetimetz"]).columns:
        df_results[col] = df_results[col].dt.tz_localize(None)

    best = df_results.sort_values("win_rate_%", ascending=False).iloc[0]

    with pd.ExcelWriter(output, engine="openpyxl") as writer:

        # Sheet 1 — All Results sorted by win rate
        df_results.sort_values(
            "win_rate_%", ascending=False
        ).to_excel(writer, sheet_name="All Results", index=False)

        # Sheet 2 — Top 20 Win Rate
        df_results.nlargest(20, "win_rate_%").to_excel(
            writer, sheet_name="Top 20 Win Rate", index=False)

        # Sheet 3 — Top 20 Sharpe
        df_results.nlargest(20, "sharpe_ratio").to_excel(
            writer, sheet_name="Top 20 Sharpe", index=False)

        # Sheet 4 — Top 20 Final Capital
        df_results.nlargest(20, "final_capital_₹").to_excel(
            writer, sheet_name="Top 20 Capital", index=False)

        # Sheet 5 — Top 20 CAGR
        df_results.nlargest(20, "cagr_%").to_excel(
            writer, sheet_name="Top 20 CAGR", index=False)

        # Sheet 6 — Parameter Impact
        impact_df.to_excel(
            writer, sheet_name="Parameter Impact", index=False)

        # Sheet 7 — Best Combo Summary
        summary = pd.DataFrame({
            "Item": [
                "── RUN INFO ──────────────────",
                "Total Combinations Run",
                "Valid Results",
                "",
                "── BEST COMBO (by Win Rate) ──",
                "♔ king_cash_guard",
                "♔ king_total_stop",
                "♝ stop_loss",
                "♝ atr_multiplier",
                "♝ max_drawdown",
                "♜ max_positions",
                "♜ top_alloc_%",
                "♟ promotion_at_%",
                "♞ nifty_drop_trigger",
                "♞ hedge_alloc_%",
                "",
                "── BEST COMBO RESULTS ─────────",
                "Win Rate %",
                "Final Capital ₹",
                "Total Return %",
                "CAGR %",
                "Sharpe Ratio",
                "Max Drawdown %",
                "Risk/Reward",
                "Total Trades",
                "Avg Hold Days",
                "Promoted Win Rate %",
                "Total Hedge P&L ₹",
            ],
            "Value": [
                "",
                total_combos,
                len(df_results),
                "",
                "",
                best["♔ king_cash_guard"],
                best["♔ king_total_stop"],
                best["♝ stop_loss"],
                best["♝ atr_multiplier"],
                best["♝ max_drawdown"],
                best["♜ max_positions"],
                best["♜ top_alloc_%"],
                best["♟ promotion_at_%"],
                best["♞ nifty_drop_trigger"],
                best["♞ hedge_alloc_%"],
                "",
                "",
                best["win_rate_%"],
                best["final_capital_₹"],
                best["total_return_%"],
                best["cagr_%"],
                best["sharpe_ratio"],
                best["max_drawdown_%"],
                best["risk_reward"],
                best["total_trades"],
                best["avg_hold_days"],
                best["promoted_win_rate_%"],
                best["total_hedge_pnl_₹"],
            ]
        })
        summary.to_excel(writer, sheet_name="Best Combo Summary", index=False)

    print(f"✅ Saved: {output}")

# ═══════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════
if __name__ == "__main__":

    # 1. Load tickers + raw data
    tickers  = load_nse_tickers()
    print(f"♟ Tickers: {len(tickers)}\n")

    raw_data = load_raw_data(tickers)
    nifty    = load_nifty()

    if not raw_data:
        print("ERROR: No data loaded."); exit(1)

    # 2. Pre-compute signals for each unique atr_period
    #    (avoids recomputing signals for every combo)
    print("♛ Pre-computing Queen signals for each ATR period...")
    signal_cache = {}
    for atr_p in ATR_PERIOD_LIST:
        print(f"   ATR period = {atr_p}...")
        processed = {}
        for ticker, df in raw_data.items():
            try:
                df2 = compute_utbot(df, atr_p)
                processed[ticker] = df2
            except: pass
        signal_cache[atr_p] = processed
        print(f"   ✅ {len(processed)} stocks processed")
    print()

    # 3. Shared date list
    sample_data = list(signal_cache.values())[0]
    all_dates   = sorted(set(
        d for df in sample_data.values() for d in df.index
    ))
    print(f"📅 Trading days: {len(all_dates)}\n")

    # 4. Build param grid (excluding atr_period — handled via signal_cache)
    param_grid = list(product(
        KING_CASH_GUARD_LIST,
        KING_TOTAL_STOP_LIST,
        STOP_LOSS_LIST,
        ATR_MULT_LIST,
        MAX_DRAWDOWN_LIST,
        MAX_POSITIONS_LIST,
        TOP_ALLOC_LIST,
        PROMOTION_LIST,
        NIFTY_DROP_LIST,
        HEDGE_ALLOC_LIST,
        ATR_PERIOD_LIST,
    ))

    print(f"🚀 Running {len(param_grid)} combinations...\n")

    results = []
    t0      = time.time()

    for i, (
        kcg, kts, sl, atr, mdd,
        mp, ta, promo, ndt, ha, atr_p
    ) in enumerate(param_grid):

        # Use pre-computed signals for this atr_period
        all_data = signal_cache[atr_p]

        res = run_single(
            all_data, all_dates, nifty,
            kcg, kts, sl, atr, mdd,
            mp, min(mp-1, 5), ta,
            promo, ndt, ha,
        )
        if res:
            results.append(res)

        if (i+1) % 20 == 0 or (i+1) == len(param_grid):
            elapsed = time.time() - t0
            rate    = (i+1) / elapsed if elapsed > 0 else 1
            eta     = (len(param_grid) - i - 1) / rate / 3600
            pct     = (i+1) / len(param_grid) * 100
            print(f"  [{i+1}/{len(param_grid)}]  {pct:.0f}%  |  "
                  f"valid: {len(results)}  |  "
                  f"elapsed: {elapsed/3600:.1f}hr  |  "
                  f"ETA: {eta:.1f}hr  |  "
                  f"speed: {rate*3600:.0f}/hr")

    total_time = (time.time() - t0) / 3600
    print(f"\n✅ Done in {total_time:.1f} hrs — valid: {len(results)}/{len(param_grid)}\n")

    if not results:
        print("No valid results."); exit(1)

    df_results = pd.DataFrame(results)
    impact_df  = build_impact(df_results)

    # 5. Print top 5 by win rate
    print("🏆 TOP 5 BY WIN RATE:")
    cols = [
        "♔ king_cash_guard","♝ stop_loss","♝ atr_multiplier",
        "♜ max_positions","♟ promotion_at_%",
        "win_rate_%","sharpe_ratio","cagr_%","final_capital_₹"
    ]
    print(df_results.nlargest(5,"win_rate_%")[cols].to_string(index=False))

    print("\n📊 PARAMETER IMPACT (sorted by avg win rate):")
    print(
        impact_df[["Parameter","Value","Avg Win Rate %",
                   "Avg Sharpe","Avg CAGR %","Avg Risk/Reward"]]
        .to_string(index=False)
    )

    # 6. Save Excel
    save_excel(df_results, impact_df)